# Features and Overfitting: How a Model Memorizes, and How L1 and L2 Stop It

The full code of [Features and Overfitting: How a Model Memorizes, and How L1 and L2 Stop It](https://residual-stream.lvtan71.workers.dev/posts/features-overfitting-regularization/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/features-overfitting-regularization) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/features-overfitting-regularization.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "features-overfitting-regularization"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `overfit.py`

The logistic-regression router on bag-of-words features, trained with no penalty for 25,600 steps on split 0: training loss, validation loss and the weight norm as it goes. Also defines `sigmoid`, `log_loss` and `train()` (with L2 or L1), which the other scripts import.

In [ ]:
!{sys.executable} overfit.py

The post printed:

```text
379 words, 120 training questions
step   100: train loss 0.2138  val loss 0.3951  |w|  4.02
step   400: train loss 0.0701  val loss 0.3196  |w|  8.32
step  1600: train loss 0.0175  val loss 0.2880  |w| 13.43
step  3200: train loss 0.0086  val loss 0.2870  |w| 16.01
step  6400: train loss 0.0043  val loss 0.2918  |w| 18.56
step 25600: train loss 0.0011  val loss 0.3117  |w| 23.64
```

## `sweep.py`

L2 and L1 at several strengths, each averaged over 20 random splits: training accuracy, validation loss and accuracy, nonzero weights.

In [ ]:
!{sys.executable} sweep.py

The post printed:

```text
l2 λ=0.0   : train acc 1.000  val loss 0.480 ± 0.034  val acc 0.818  nonzero weights 383
l2 λ=0.001 : train acc 1.000  val loss 0.421 ± 0.021  val acc 0.804  nonzero weights 383
l2 λ=0.003 : train acc 1.000  val loss 0.416 ± 0.015  val acc 0.794  nonzero weights 383
l2 λ=0.01  : train acc 0.987  val loss 0.435 ± 0.010  val acc 0.784  nonzero weights 383
l2 λ=0.03  : train acc 0.887  val loss 0.473 ± 0.006  val acc 0.753  nonzero weights 383
l2 λ=0.1   : train acc 0.751  val loss 0.516 ± 0.002  val acc 0.750  nonzero weights 383
l1 λ=0.001 : train acc 1.000  val loss 0.530 ± 0.029  val acc 0.790  nonzero weights 112
l1 λ=0.004 : train acc 1.000  val loss 0.514 ± 0.022  val acc 0.767  nonzero weights 51
l1 λ=0.008 : train acc 0.893  val loss 0.502 ± 0.012  val acc 0.768  nonzero weights 27
l1 λ=0.016 : train acc 0.772  val loss 0.532 ± 0.005  val acc 0.742  nonzero weights 10
```

## `l1_words.py`

The words L1 keeps on split 0 at λ = 0.008, and their weights.

In [ ]:
!{sys.executable} l1_words.py

The post printed:

```text
30 of 379 weights are nonzero; those of size >= 0.01:
toward serving: batching +1.70, to +1.60, token +1.34, prefix +1.26, model +1.16, memory +1.14, decode +0.91, my +0.81, tokens +0.80, draft +0.74, one +0.68, vllm +0.63, request +0.61, for +0.50, does +0.33, on +0.28, throughput +0.22, cache +0.22, time +0.12
away:           agent -1.80, need -0.87, what -0.74, attention -0.39, how -0.37, should -0.12, layer -0.02
```

## `compare.py`

The hand-picked serving-keyword count against the bag of words, and both together, on the same 20 splits.

In [ ]:
!{sys.executable} compare.py

The post printed:

```text
keyword count                   val loss 0.381  val acc 0.878
keyword count, seen words only  val loss 0.436  val acc 0.851
bag of words                    val loss 0.416  val acc 0.794
both                            val loss 0.347  val acc 0.866
```

## `honest.py`

Paired differences between λ settings over the 20 splits, with the naive and the Nadeau–Bengio corrected standard errors.

In [ ]:
!{sys.executable} honest.py

The post printed:

```text
L2 0.003 - no penalty (4,000 steps)   -0.064   naive SE 0.020   corrected SE 0.055
L2 0.003 - no penalty (16,000 steps)  -0.131   naive SE 0.030   corrected SE 0.082
L2 0.003 - L2 0.001                   -0.005   naive SE 0.006   corrected SE 0.017
L2 0.003 - L2 0.01                    -0.019   naive SE 0.006   corrected SE 0.016
```

## `peek.py`

Choosing λ on the test set instead of the validation set, and the test loss each choice reports.

In [ ]:
!{sys.executable} peek.py

The post printed:

```text
test loss, λ chosen on test: 0.403; chosen on validation: 0.439
```